# SNN Step 2 Sweep — Colab Runner
**NAND + XOR, h=20, K=[1,2,3,4,6,8,12,16], 5 conditions, 50 epochs**

Steps:
1. Mount Google Drive (断连保护)
2. Clone repo & install deps
3. Smoke test
4. Full sweep (结果实时同步到 Drive)
5. 下载结果

In [ ]:
# 1. Check GPU
!nvidia-smi

In [ ]:
# 2. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

import os
DRIVE_DIR = '/content/drive/MyDrive/SNN_step2_runs'
os.makedirs(DRIVE_DIR, exist_ok=True)
print(f'Drive mounted. Results will sync to: {DRIVE_DIR}')

In [ ]:
# 3. Clone repo
!git clone https://github.com/KunpengXu-04/SNN-async-delays.git
%cd SNN-async-delays/snn_async_delays

# Symlink runs/ -> Drive, so every completed run is saved immediately
import os
os.makedirs('/content/drive/MyDrive/SNN_step2_runs', exist_ok=True)
if not os.path.exists('runs'):
    os.symlink('/content/drive/MyDrive/SNN_step2_runs', 'runs')
else:
    # runs/ already exists from a previous session — reuse it
    import shutil
    shutil.copytree('runs', '/content/drive/MyDrive/SNN_step2_runs', dirs_exist_ok=True)
    shutil.rmtree('runs')
    os.symlink('/content/drive/MyDrive/SNN_step2_runs', 'runs')

print('runs/ -> Google Drive symlink created.')
print('Existing runs on Drive:', os.listdir('runs'))

In [ ]:
# 4. Install dependencies
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu121 -q
!pip install pyyaml matplotlib numpy -q
print('Dependencies installed.')

In [ ]:
# 5. Smoke test (XOR, K=1, 10 epochs) — should finish in ~30s
!python -m scripts.run_step2 \
    --config configs/step2_multiquery_sameop.yaml \
    --op XOR --K 1 --epochs 10 --device cuda

In [ ]:
# 6. Full sweep — 2 ops x 5 conditions x 8 K values = 80 runs
# Results are written directly to Google Drive after each run.
# If the session disconnects, re-run cells 1-4 to reconnect,
# then re-run this cell — already-completed runs will be skipped
# automatically (run dirs already exist on Drive).
!python -m scripts.run_step2 \
    --config configs/step2_multiquery_sameop.yaml \
    --sweep --device cuda

In [ ]:
# 7. Preview summary
import json
with open('runs/step2_sweep_summary.json') as f:
    summary = json.load(f)

for op in summary:
    print(f'\n=== {op} ===')
    for cname, data in summary[op].items():
        mk = data['max_K_by_tau']
        print(f'  {cname}: maxK@0.95={mk.get("0.95")}, maxK@0.90={mk.get("0.90")}, maxK@0.85={mk.get("0.85")}')

In [ ]:
# 8. Package results and download
import shutil, os

items_to_zip = [
    'runs/step2_sweep_summary.json',
    'runs/step2_plots_NAND',
    'runs/step2_plots_XOR',
]
for d in os.listdir('runs'):
    if d.startswith('step2_') and os.path.isdir(f'runs/{d}'):
        items_to_zip.append(f'runs/{d}')

os.makedirs('export', exist_ok=True)
for item in items_to_zip:
    if os.path.exists(item):
        dest = f'export/{item}'
        os.makedirs(os.path.dirname(dest), exist_ok=True)
        if os.path.isdir(item):
            shutil.copytree(item, dest, dirs_exist_ok=True)
        else:
            shutil.copy2(item, dest)

shutil.make_archive('step2_results', 'zip', 'export')
print('step2_results.zip ready.')

# Also copy zip to Drive as backup
shutil.copy2('step2_results.zip', '/content/drive/MyDrive/SNN_step2_runs/step2_results.zip')
print('Zip also saved to Google Drive.')

In [ ]:
# 9. Download zip directly to your computer
from google.colab import files
files.download('step2_results.zip')